In [40]:
import numpy as np
import random
import math

In [41]:
x = 2.0
w = -3.0
b = 6.88
target = 0.5

n= x * w + b
o = math.tanh(n)
L = (o - target)**2

print(f"n: {n:.4f}")
print(f"o: {o:.4f}")
print(f"Loss: {L:.4f}")

n: 0.8800
o: 0.7064
Loss: 0.0426


dL/dw = ?

dL/do = 2 * (tanh(n) - target)

do/dw = (1 - tanh(n)^2) * x


-> dL/dw = 2 * x * (1 - tanh(n)^2) * (tanh(n) - target)

dL/dx = ?

dL/do = 2 * (tanh(n) - target)

do/dx = (1 - tanh(n)^2) * w

-> dL/dx = 2 * w * (1 - tanh(n)^2) * (tanh(n) - target)

In [42]:
dL_do = 2 * (o - target)
do_dw = (1 - o**2) * x
do_dx = (1 - o**2) * w

dL_dw_analytical = dL_do * do_dw
dL_dx_analytical = dL_do * do_dx

print(f"Analytical dL/dw: {dL_dw_analytical:.8f}")
print(f"Analytical dL/dx: {dL_dx_analytical:.8f}")

Analytical dL/dw: 0.41364099
Analytical dL/dx: -0.62046148


In [43]:
h=0.0001

n2 = (x * (w + h)) + b
o2 = math.tanh(n2)
L2 = (o2 - target)**2

print(f"Numerical dL/dw: {(L2 - L)/h}")

n3 = ((x + h) * w) + b
o3 = math.tanh(n3)
L3 = (o3 - target)**2
print(f"Numerical dL/dx: {(L3 - L)/h}")


Numerical dL/dw: 0.4136829102058953
Numerical dL/dx: -0.6203670111952497


In [62]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = math.tanh(x)
        out = Value(t, (self, ), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data**(other - 1)) * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self * other

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __neg__(self):
        return self * -1

    def __rsub__(self, other):
        return other + (-self)

In [63]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
d = Value(-2.0)

e = a * b
f = e + c
g = f * d
L = g**2

L.backward()
print(f"Forward Pass Result (L): {L.data}")
print(f"dL/da: {a.grad:>10.2f}")
print(f"dL/db: {b.grad:>10.2f}")
print(f"dL/dc: {c.grad:>10.2f}")
print(f"dL/dd: {d.grad:>10.2f}")

Forward Pass Result (L): 64.0
dL/da:     -96.00
dL/db:      64.00
dL/dc:      32.00
dL/dd:     -64.00


In [64]:
import torch

x1 = torch.Tensor([2.0]).double()    ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()    ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()   ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()    ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()   ; b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

0.7071066904050358
---
x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [65]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self,x):
        outs = [n(x) for n in self.neurons]
        return outs


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

In [66]:
n = MLP(3, [4, 4, 1])
xs =[
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]
ys = [1.0, -1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]
ypred

[Value(data=-0.813152376489648, grad=0.0),
 Value(data=-0.7924787939223708, grad=0.0),
 Value(data=-0.771312948369127, grad=0.0),
 Value(data=-0.7946331500270245, grad=0.0)]

In [68]:
loss = sum((yo - yp)**2 for yo, yp in zip(ys, ypred))
loss

Value(data=6.603592502101514, grad=0.0)